<a href="https://colab.research.google.com/github/SIDHANTMAHI/AI_-safety_project-/blob/main/project_1/truthfulqa_qwen3_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q transformers pandas tqdm

In [26]:
import os
import gc
import torch
import pandas as pd
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [27]:
MODEL_ID = "Qwen/Qwen3-4B-Base"
# MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

DEVICE = "cuda"


DRIVE_BASE_PATH = "/content/drive/MyDrive/aisa_miniproject"

In [28]:
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
MODEL = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [21]:
#   FOLLOWING COMMENTED CODE IS FOR REASONING MODEL.

# def generate_response(user_prompt):

#     if TOKENIZER.chat_template is not None:
#         chat_messages = [{"role": "user", "content": user_prompt}]
#         formatted_text = TOKENIZER.apply_chat_template(
#             chat_messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )
#     else:

#         formatted_text = user_prompt


#     tokenized_inputs = TOKENIZER([formatted_text], return_tensors="pt").to(DEVICE)


#     output_token_ids = MODEL.generate(
#         **tokenized_inputs,
#         max_new_tokens=2048
#     )
#     response_token_ids = output_token_ids[0][len(tokenized_inputs.input_ids[0]):].tolist()


#     try:
#         think_end_index = len(response_token_ids) - response_token_ids[::-1].index(151668)
#     except ValueError:
#         think_end_index = 0

#     reasoning_trace = TOKENIZER.decode(
#         response_token_ids[:think_end_index], skip_special_tokens=True
#     ).strip("\n")

#     final_answer = TOKENIZER.decode(
#         response_token_ids[think_end_index:], skip_special_tokens=True
#     ).strip("\n")

#     return reasoning_trace, final_answer



# THIS IS FOR INSTRCUT MODEL

# def generate_response(user_prompt):
#     chat_messages = [{"role": "user", "content": user_prompt}]
#     formatted_text = TOKENIZER.apply_chat_template(
#         chat_messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     tokenized_inputs = TOKENIZER([formatted_text], return_tensors="pt").to(DEVICE)

#     output_token_ids = MODEL.generate(
#         **tokenized_inputs,
#         max_new_tokens=2048
#     )
#     response_token_ids = output_token_ids[0][len(tokenized_inputs.input_ids[0]):].tolist()

#     final_answer = TOKENIZER.decode(
#         response_token_ids, skip_special_tokens=True
#     ).strip("\n")

#     return "", final_answer

# FOLLOWING CODE FOR BASE MODEL

def generate_response(user_prompt):
    formatted_text = f"Q: {user_prompt}\nA:"

    tokenized_inputs = TOKENIZER([formatted_text], return_tensors="pt").to(DEVICE)

    output_token_ids = MODEL.generate(
        **tokenized_inputs,
        max_new_tokens=256,
        stop_strings=["\nQ:"],
        tokenizer=TOKENIZER
    )
    response_token_ids = output_token_ids[0][len(tokenized_inputs.input_ids[0]):].tolist()

    final_answer = TOKENIZER.decode(
        response_token_ids, skip_special_tokens=True
    ).split("\nQ:")[0].strip()

    return "", final_answer

In [24]:

CHECKPOINT_INDEX = -1

# Create output directory in Google Drive
output_dir = f"{DRIVE_BASE_PATH}/results/{MODEL_ID}"
os.makedirs(output_dir, exist_ok=True)

if CHECKPOINT_INDEX >= 0:
    checkpoint_path = f"{output_dir}/tqa_checkpoint_{CHECKPOINT_INDEX:04d}.csv"
    print(f"Loading checkpoint: {checkpoint_path}")
    filtered_questions = pd.read_csv(checkpoint_path)
else:
    # Download TruthfulQA
    truthfulqa_dataframe = pd.read_csv(
        "https://raw.githubusercontent.com/sylinrl/TruthfulQA/"
        "refs/heads/main/data/v1/TruthfulQA.csv"
    )
    filtered_questions = truthfulqa_dataframe[
        truthfulqa_dataframe["Category"] == "Misconceptions"
    ]["Question"].copy().reset_index()
    print(f"Loaded {len(filtered_questions)} questions.")

Loaded 100 questions.


In [29]:
for question_idx in tqdm(range(CHECKPOINT_INDEX + 1, filtered_questions.shape[0])):
    question_text = filtered_questions.loc[question_idx, "Question"]
    reasoning_trace, final_answer = generate_response(question_text)

    filtered_questions.loc[question_idx, MODEL_ID + "_think"] = reasoning_trace
    filtered_questions.loc[question_idx, MODEL_ID + "_answer"] = final_answer

    # Save checkpoint to Google Drive
    checkpoint_save_path = f"{output_dir}/tqa_checkpoint_{question_idx:04d}.csv"
    filtered_questions.to_csv(checkpoint_save_path, index=False)

  0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
del MODEL
del TOKENIZER
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared. Ready for next model.")

GPU memory cleared. Ready for next model.
